## DEG Analysis

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load adata
adata_path = "/media/rokny/DATA2/Sally/data/srt/Mouse_embryo/E9.5_E1S1.MOSTA.h5ad-20240116T040300Z-001/E9.5_E1S1.MOSTA.h5ad"

adata = sc.read_h5ad(adata_path)

print(adata)

# Load Leiden cluster labels 
labels_path = "/media/rokny/DATA2/Sally/CellPLM/benchmarking_results/cellplm_continualpretrain_clusters_ME.npz"

data = np.load(labels_path, allow_pickle=True)
arrays = {key: data[key] for key in data.files}

cluster_labels = arrays['labels']

adata.obs['leiden'] = cluster_labels
adata.obs['leiden'] = adata.obs['leiden'].astype('category')

gt_column = 'annotation'

# Logarithmise data if raw 
sc.pp.log1p(adata) 


In [ ]:
top = 20
dataset = 'ME'
model = 'CellPLM'

# Perform differential expression analysis to get the top genes for each ground truth cell type
sc.tl.rank_genes_groups(adata, groupby=gt_column, method='wilcoxon', use_raw=False, n_genes=top)

# Convert to DataFrame
rank_gt = sc.get.rank_genes_groups_df(adata, None)  # all groups
# Build dictionary of top genes per celltype
top_genes_gt = (
    rank_gt.groupby("group")["names"]
    .apply(lambda x: x.head(top).tolist())
    .to_dict()
)

for k, v in top_genes_gt.items():
    print(f"{k}\t{', '.join(v)}")


In [ ]:
# Perform differential expression analysis for predicted clusters (e.g., 'leiden' or 'model_cluster')
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon', use_raw=False, n_genes=top)

# Extract the results as a Pandas DataFrame
rank_genes = sc.get.rank_genes_groups_df(adata, None)  # all groups

# Create dictionary of top genes per cluster
top_genes_pred = (
    rank_genes.groupby("group")["names"]
    .apply(lambda x: x.head(top).tolist())
    .to_dict()
)

for k, v in top_genes_pred.items():
    print(f"{k}\t{', '.join(v)}")

In [ ]:
# Compare top genes between predicted clusters and ground truth cell types
similarity_matrix = {}

for cluster, pred_genes in top_genes_pred.items():
    similarity_matrix[cluster] = {}
    for cell_type, gt_genes in top_genes_gt.items():
        # Find the number of matching genes
        matching_genes = len(set(pred_genes) & set(gt_genes))
        similarity_matrix[cluster][cell_type] = matching_genes

# Convert the similarity matrix to a DataFrame for easier viewing
similarity_df = pd.DataFrame(similarity_matrix)

print("Similarity Matrix (number of matching genes):")
print(similarity_df)

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcdefaults()

plt.figure(figsize=(10, 8), dpi=300)
sns.heatmap(
    similarity_df,
    annot=True,
    fmt='d', 
    cmap='Reds',
    linecolor='white',
    square=False,
    cbar_kws={'label': 'No. of Matching Genes'},
    annot_kws={"size": 12}
)

plt.xlabel("Predicted Cluster", fontsize=12)
plt.ylabel("Ground Truth Cell Type", fontsize=12)
plt.title("Gene Overlap Between Cluster and Cell Type DEGs", fontsize=14)

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.show()

file_path = f'/media/rokny/DATA2/Sally/{model}/figures/similarity_heatmap_{top}_{dataset}.svg'
plt.savefig(file_path, bbox_inches='tight', format='svg')
plt.show()

In [ ]:
def calculate_jaccard_index(genes_gt, genes_pred):
    set_gt = set(genes_gt)
    set_pred = set(genes_pred)
    intersection = len(set_gt & set_pred)
    union = len(set_gt | set_pred)
    return intersection / union if union > 0 else 0.0

jaccard_matrix = pd.DataFrame(
    index=top_genes_gt.keys(),
    columns=top_genes_pred.keys(),
    dtype=float
)

for ct, genes_gt in top_genes_gt.items():
    for cl, genes_pred in top_genes_pred.items():
        jaccard_matrix.loc[ct, cl] = calculate_jaccard_index(genes_gt, genes_pred)


In [ ]:
# Plot as heatmap
plt.figure(figsize=(10, 8), dpi=300)
sns.heatmap(jaccard_matrix, \
            annot=True, 
            fmt=".2f", 
            cmap='Blues', 
            linewidths=0.5, 
            cbar_kws={'label': 'Jaccard Index'},
            annot_kws={"size": 12})
plt.title("Pairwise Jaccard Index between Ground Truth and Predicted Cell Types", fontsize=14, pad=20)
plt.xlabel("Predicted Cluster", fontsize=12)
plt.ylabel("Ground Truth Cell Type", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()

# Save plot
file_path = f'/media/rokny/DATA2/Sally/{model}/figures/jaccard_index_pairwise_heatmap_{top}_{dataset}.svg'
plt.savefig(file_path, bbox_inches='tight')
plt.show()